# Random Quantum States and Operators

Random quantum objects are fundamental tools in quantum information science. They appear in:

- **Randomized benchmarking** -- characterizing gate errors using random Clifford sequences
- **Quantum volume** -- benchmarking processors with Haar-random circuits
- **Quantum state tomography** -- testing estimators against random states
- **Unitary $t$-designs** -- approximating the Haar measure with finite ensembles
- **Random quantum channels** -- studying typical noise properties

Quax provides efficient, JAX-compatible generators for random states, unitaries, operators, and channels, all supporting batch generation.

In [ ]:
import jax
import jax.numpy as jnp
import quax as qx

## Random state vectors

Random pure states $|\psi\rangle \in \mathbb{C}^d$ are drawn uniformly from the unit sphere in $\mathbb{C}^d$ (the Fubini-Study measure). This is achieved by generating a complex Gaussian vector and normalizing.

For qubit systems, `dims` specifies the qudit dimensions, e.g. `(2,)` for one qubit, `(2, 2)` for two qubits.

In [ ]:
key = jax.random.key(42)

# Single random qubit state
psi = qx.random_state_vector(dims=(2,), key=key)
print(f"Random qubit state: {psi}")
print(f"  Amplitudes: {psi.matrix}")
print(f"  Norm: {jnp.linalg.norm(psi.matrix):.6f}")

# Random 3-qubit state
key, subkey = jax.random.split(key)
psi_3q = qx.random_state_vector(dims=(2, 2, 2), key=subkey)
print(f"\nRandom 3-qubit state: {psi_3q}")
print(f"  Shape: {psi_3q.matrix.shape}")

In [ ]:
# Batch of random states
key, subkey = jax.random.split(key)
psi_batch = qx.random_state_vector(dims=(2,), key=subkey, size=(100,))
print(f"Batch of 100 random qubit states: {psi_batch}")

# All should be normalized
norms = jnp.linalg.norm(psi_batch.matrix, axis=-1)
print(f"  All norms approx 1: {jnp.allclose(norms, 1.0, atol=1e-6)}")

## Random density matrices

Random mixed states are generated from the Hilbert-Schmidt measure. A random density matrix of rank $r$ in dimension $d$ is constructed as:

1. Draw a $d \times r$ complex Ginibre matrix $A$
2. Form $\rho = A A^\dagger / \mathrm{Tr}[A A^\dagger]$

The **rank** parameter controls the purity:
- Rank 1: pure states ($\rho = |\psi\rangle\langle\psi|$)
- Rank $d$: generically full-rank mixed states

This distribution is related to the induced measures studied in [Zyczkowski & Sommers, J. Phys. A 34, 7111 (2001)](https://arxiv.org/abs/quant-ph/0012101).

In [ ]:
key, subkey = jax.random.split(key)

# Rank-1 density matrix (pure state)
rho_pure = qx.random_density_matrix(rank=1, dims=(2,), key=subkey)
print("Rank-1 (pure) density matrix:")
print(f"  {jnp.round(rho_pure.matrix, 4)}")
print(f"  Tr[rho^2] = {jnp.real(jnp.trace(rho_pure.matrix @ rho_pure.matrix)):.6f}  (1 = pure)")

# Rank-2 density matrix (mixed)
key, subkey = jax.random.split(key)
rho_mixed = qx.random_density_matrix(rank=2, dims=(2,), key=subkey)
print("\nRank-2 (mixed) density matrix:")
print(f"  {jnp.round(rho_mixed.matrix, 4)}")
print(f"  Tr[rho^2] = {jnp.real(jnp.trace(rho_mixed.matrix @ rho_mixed.matrix)):.6f}  (< 1 = mixed)")

In [ ]:
# Batch of 2-qubit density matrices
key, subkey = jax.random.split(key)
rho_batch = qx.random_density_matrix(rank=2, dims=(2, 2), key=subkey, size=(5, 3))
print(f"Batch of random density matrices: {rho_batch}")

# Verify all are valid density matrices: Tr[rho] = 1, rho >= 0
traces = jnp.trace(rho_batch.matrix, axis1=-2, axis2=-1)
print(f"  All traces approx 1: {jnp.allclose(traces, 1.0, atol=1e-6)}")
print(f"  All Hermitian: {qx.is_hermitian(rho_batch)}")

## Haar-random unitaries

Random unitaries drawn from the **Haar measure** are the gold standard for uniform sampling on the unitary group $U(d)$. The algorithm uses QR decomposition of a complex Ginibre matrix, following [Mezzadri, Notices of the AMS 54, 592 (2007)](https://arxiv.org/abs/math-ph/0609050):

1. Draw a $d \times d$ complex Ginibre matrix $Z$
2. Compute $Z = QR$ (QR decomposition)
3. Correct phases: $U = Q \cdot \mathrm{diag}(R_{ii} / |R_{ii}|)$

The resulting $U$ is Haar-distributed on $U(d)$.

In [ ]:
key, subkey = jax.random.split(key)

# Single Haar-random qubit unitary
U = qx.random_unitary(dims=((2,), (2,)), key=subkey)
print("Random SU(2) gate:")
print(f"  {jnp.round(U.matrix, 4)}")
print(f"  UU_dag = I: {jnp.allclose(U.matrix @ U.matrix.conj().T, jnp.eye(2), atol=1e-10)}")

# Haar-random 2-qubit unitary (SU(4))
key, subkey = jax.random.split(key)
U_2q = qx.random_unitary(dims=((2, 2), (2, 2)), key=subkey)
print(f"\nRandom SU(4) gate: {U_2q}")
print(f"  Is unitary: {qx.is_unitary(U_2q)}")

In [ ]:
# Large batch of unitaries
key, subkey = jax.random.split(key)
ensemble = qx.random_unitary(dims=((2,), (2,)), key=subkey, size=(1000,))
print(f"Ensemble of {ensemble.matrix.shape[0]} unitaries: {ensemble}")

# Verify all are unitary
print(f"  All unitary: {qx.is_unitary(ensemble)}")

## Unitary designs

A **unitary $t$-design** is a finite ensemble of unitaries $\{U_i\}$ that reproduces the first $t$ moments of the Haar measure. Informally:

- **1-design:** The ensemble average of $U \rho U^\dagger$ equals the maximally mixed state for all $\rho$
- **2-design:** The ensemble reproduces the second moment, which is needed for randomized benchmarking

The single-qubit Clifford group (24 elements) is an exact 3-design. Quax can test the design property of any ensemble.

In [ ]:
# The Clifford ensemble is a 2-design (and in fact a 3-design)
cliffords = qx.ensembles.CLIFFORD_ENSEMBLE
print(f"Clifford ensemble: {cliffords.matrix.shape[0]} unitaries")
print(f"  Is 1-design: {qx.is_one_design(cliffords, atol=1e-6)}")
print(f"  Is 2-design: {qx.is_two_design(cliffords, atol=1e-6)}")

# The tetrahedral ensemble (12 elements) is also a 2-design
tetra = qx.ensembles.TETRAHEDRAL_ENSEMBLE
print(f"\nTetrahedral ensemble: {tetra.matrix.shape[0]} unitaries")
print(f"  Is 1-design: {qx.is_one_design(tetra, atol=1e-6)}")
print(f"  Is 2-design: {qx.is_two_design(tetra, atol=1e-6)}")

In [ ]:
# A large Haar-random ensemble is approximately a 2-design
key, subkey = jax.random.split(key)
large_ensemble = qx.random_unitary(dims=((2,), (2,)), key=subkey, size=(10000,))
print(f"Large Haar ensemble ({large_ensemble.matrix.shape[0]} unitaries):")
print(f"  Is 1-design: {qx.is_one_design(large_ensemble, atol=1e-1)}")
print(f"  Is 2-design: {qx.is_two_design(large_ensemble, atol=1e-1)}")

# Smaller ensemble: not a good design
key, subkey = jax.random.split(key)
small_ensemble = qx.random_unitary(dims=((2,), (2,)), key=subkey, size=(5,))
print(f"\nSmall ensemble ({small_ensemble.matrix.shape[0]} unitaries):")
print(f"  Is 1-design: {qx.is_one_design(small_ensemble, atol=1e-1)}")
print(f"  Is 2-design: {qx.is_two_design(small_ensemble, atol=1e-1)}")

## Random operators and observables

Beyond unitaries and states, Quax supports:

- `random_operator` -- general (non-unitary) linear maps drawn from the complex Ginibre ensemble
- `random_observable` -- Hermitian operators, constructed by symmetrizing a Ginibre matrix: $A = (G + G^\dagger)/2$

These are useful for testing code against generic inputs and for constructing random Hamiltonians.

In [ ]:
key, subkey = jax.random.split(key)

# Random general operator
op = qx.random_operator(dims=((2,), (2,)), key=subkey)
print("Random operator:")
print(f"  {jnp.round(op.matrix, 4)}")
print(f"  Is unitary: {qx.is_unitary(op)}")
print(f"  Is Hermitian: {qx.is_hermitian(op)}")

# Random Hermitian observable
key, subkey = jax.random.split(key)
obs = qx.random_observable(dims=((2,), (2,)), key=subkey)
print("\nRandom observable (Hermitian):")
print(f"  {jnp.round(obs.matrix, 4)}")
print(f"  Is Hermitian: {qx.is_hermitian(obs)}")
print(f"  Eigenvalues: {jnp.round(jnp.linalg.eigvalsh(obs.matrix), 4)}")

In [ ]:
# Non-square operator: map from 2 qubits to 1 qubit
key, subkey = jax.random.split(key)
rect_op = qx.random_operator(dims=((2,), (2, 2)), key=subkey)
print("Non-square operator (2x4):")
print(f"  Matrix shape: {rect_op.matrix.shape}")
print(f"  dims: {rect_op.dims}")

## Random quantum channels (Choi)

Random CPTP (completely positive, trace-preserving) channels are generated via the BCSZ distribution [Bruzda et al., Phys. Lett. A 373, 320 (2009)](https://arxiv.org/abs/0804.2361). The construction:

1. Generate a $d^2 \times r$ Ginibre matrix $X$ (where $r$ is the Kraus rank)
2. Form $J = X X^\dagger$ 
3. Enforce trace preservation by a partial trace constraint

The **rank** parameter determines the number of Kraus operators:
- Rank 1: unitary channel
- Rank $d^2$: maximally mixed channel (generically)

In [ ]:
key, subkey = jax.random.split(key)

# Random single-qubit CPTP channel
choi = qx.random_choi(dims=((2,), (2,)), rank=4, key=subkey)
print("Random CPTP channel (Choi matrix):")
print(f"  {jnp.round(jnp.real(choi.matrix), 4)}")

# Verify CPTP properties
print(f"\n  Is CPTP: {qx.is_cptp(choi)}")
print(f"  Is CP: {qx.is_completely_positive(choi)}")
print(f"  Is TP: {qx.is_trace_preserving(choi)}")

# Choi eigenvalues (all non-negative for CP)
eigvals = jnp.linalg.eigvalsh(choi.matrix)
print(f"  Choi eigenvalues: {jnp.round(eigvals, 6)}")

In [ ]:
# Convert to other representations
kraus = qx.choi_to_kraus(choi)
pl = qx.choi_to_pauli_liouville(choi)
superop = qx.choi_to_superop(choi)

print(f"Kraus: {kraus.matrix.shape[0]} operators")
print(f"Pauli-Liouville:\n{jnp.round(jnp.real(pl.matrix), 4)}")
print(f"\nProcess fidelity: {qx.process_fidelity(choi):.4f}")

In [ ]:
# Batch of random channels
key, subkey = jax.random.split(key)
choi_batch = qx.random_choi(dims=((2,), (2,)), rank=2, key=subkey, size=(50,))
print(f"Batch of {choi_batch.matrix.shape[0]} random channels: {choi_batch}")

# Compute process fidelities for the batch
fids = jnp.array([qx.process_fidelity(qx.Choi.from_matrix(c, choi_batch.dims)) for c in choi_batch.matrix])
print(f"  Process fidelity range: [{fids.min():.4f}, {fids.max():.4f}]")
print(f"  Mean process fidelity:  {fids.mean():.4f}")

## Random qudit states and operators

All random generators in Quax support arbitrary qudit dimensions — simply change the `dims` tuple. This includes qutrits ($d=3$), quarts ($d=4$), and mixed-dimension registers like qubit-qutrit systems (`dims=(2, 3)`).

Random qutrit state vectors are drawn uniformly from the unit sphere in $\mathbb{C}^3$, random qutrit unitaries are Haar-distributed on $U(3)$, and random qutrit channels follow the BCSZ distribution on the space of $3 \times 3$ CPTP maps.

In [ ]:
key, subkey = jax.random.split(key)

# Random qutrit state vector (uniform on the unit sphere in C^3)
psi_qt = qx.random_state_vector(dims=(3,), key=subkey)
print(f"Random qutrit state: {psi_qt}")
print(f"  Amplitudes: {jnp.round(psi_qt.matrix, 4)}")

# Random qutrit density matrix
key, subkey = jax.random.split(key)
rho_qt = qx.random_density_matrix(rank=3, dims=(3,), key=subkey)
print(f"\nRandom qutrit density matrix (rank 3): {rho_qt}")
print(f"  Purity Tr[rho^2] = {jnp.real(jnp.trace(rho_qt.matrix @ rho_qt.matrix)):.4f}")

# Random qutrit unitary (Haar on U(3))
key, subkey = jax.random.split(key)
U_qt = qx.random_unitary(dims=((3,), (3,)), key=subkey)
print(f"\nRandom SU(3) unitary: {U_qt}")
print(f"  Is unitary: {qx.is_unitary(U_qt)}")

# Random qutrit CPTP channel
key, subkey = jax.random.split(key)
choi_qt = qx.random_choi(dims=((3,), (3,)), rank=9, key=subkey)
print(f"\nRandom qutrit channel: {choi_qt}")
print(f"  Is CPTP: {qx.is_cptp(choi_qt)}")

In [ ]:
# Mixed-dimension register: qubit-qutrit random states
key, subkey = jax.random.split(key)
psi_23 = qx.random_state_vector(dims=(2, 3), key=subkey)
print(f"Random qubit-qutrit state: {psi_23}")
print(f"  dims: {psi_23.dims}, data shape: {psi_23.data.shape}")

# Batch of random quart (d=4) density matrices
key, subkey = jax.random.split(key)
rho_quart = qx.random_density_matrix(rank=4, dims=(4,), key=subkey, size=(10,))
print(f"\nBatch of quart density matrices: {rho_quart}")

# Two-qutrit random unitary (Haar on U(9))
key, subkey = jax.random.split(key)
U_2qt = qx.random_unitary(dims=((3, 3), (3, 3)), key=subkey)
print(f"\nRandom 2-qutrit unitary: {U_2qt}")
print(f"  Is unitary: {qx.is_unitary(U_2qt)}")

## The Ginibre ensemble

The **complex Ginibre ensemble** is the foundational distribution underlying all the random objects above. A $d \times k$ Ginibre matrix has entries independently drawn as

$$G_{ij} \sim \mathcal{N}(0,1) + i\,\mathcal{N}(0,1)$$

This low-level function is exposed as `qx.ginibre_matrix_complex` for advanced use cases.

In [ ]:
key, subkey = jax.random.split(key)

# Raw Ginibre matrix
G = qx.ginibre_matrix_complex(dim=4, k=3, key=subkey)
print(f"Ginibre matrix shape: {G.shape}")
print(f"  Mean magnitude: {jnp.abs(G).mean():.4f}")
print(f"  Complex entries: dtype={G.dtype}")

# Batch of Ginibre matrices
key, subkey = jax.random.split(key)
G_batch = qx.ginibre_matrix_complex(dim=4, k=4, key=subkey, size=(100,))
print(f"\nBatch of Ginibre matrices: {G_batch.shape}")